<a href="https://colab.research.google.com/github/Silva015/tcc-2026/blob/main/PureKAN_EntradaSaida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KAN PURO ENTRADA SAÍDA

In [ ]:
# Instalação da implementação eficiente da rede KAN e do Albumentations
!pip install git+https://github.com/Blealtan/efficient-kan -q
!pip install albumentations -q

# Clonagem dos repositórios contendo os dados e códigos base
!git clone https://gitlab.com/lisa-unb/leguwoi.git -q
!git clone https://github.com/pedrogarciafreitas/FDCPUnBGunshotDB.git -q

import os
from glob import glob
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.models as models
from efficient_kan import KAN
import copy
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# Imports do Albumentations para Data Augmentation Avançado
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def inspecionar_diretorio(caminho_base):
    print(f"--- RAIO-X DO DIRETÓRIO: {caminho_base} ---")
    for raiz, diretorios, arquivos in os.walk(caminho_base):
        if '.git' not in raiz:
            print(f"📂 Pasta: {raiz}")
            print(f"   -> Contém {len(arquivos)} arquivos e {len(diretorios)} subpastas.")
            if len(arquivos) > 0:
                print(f"   -> Amostras: {arquivos[:3]}\n")

inspecionar_diretorio('./FDCPUnBGunshotDB')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
--- RAIO-X DO DIRETÓRIO: ./FDCPUnBGunshotDB ---
📂 Pasta: ./FDCPUnBGunshotDB
   -> Contém 3 arquivos e 3 subpastas.
   -> Amostras: ['README.md', 'CODES.pdf', 'CODIGOS.pdf']

📂 Pasta: ./FDCPUnBGunshotDB/database
   -> Contém 0 arquivos e 2 subpastas.
📂 Pasta: ./FDCPUnBGunshotDB/database/ENTRADAS_EQX
   -> Contém 0 arquivos e 3 subpastas.
📂 Pasta: ./FDCPUnBGunshotDB/database/ENTRADAS_EQX/CURTA
   -> Contém 232 arquivos e 0 subpastas.
   -> Amostras: ['2016.1221SH17.38PCTEWNTTYCXSN02F08_EQX.JPG', '2020.0159SH21.38PCTEWSTTYCXSN01F02_EQX.JPG', '2017.0829SH19.38PCTEWSTTYCXSN04F02_EQX.JPG']

📂 Pasta: ./FDCPUnBGunshotDB/database/ENTRADAS_EQX/ENCOSTADO
   -> Contém 42 arquivos e 0 subpastas.
   -> Amostras: ['2016.0576SH55.40PCTEWSTTYEXSN01F03_EQX.JPG', '2016.0600SH24.32PCTEWSTTYEXSN01F03_EQX.JPG', '2016.0576SH55.40PCTEWSTTYEXSN01F04_EQX.JPG']

📂 Pasta: ./FDCPUn

In [ ]:
class GunshotDataset(Dataset):
    """
    Carrega as imagens de ferimentos de arma de fogo e atribui as labels corretas.
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        print("🔄 Mapeando imagens da base de dados...")

        # 1. Mapeamento das imagens de ENTRADA (Label 0)
        caminho_entradas = os.path.join(root_dir, 'database', 'ENTRADAS_EQX', '**', '*.JPG')
        for path in glob(caminho_entradas, recursive=True):
            self.image_paths.append(path)
            self.labels.append(0)

        # 2. Mapeamento das imagens de SAÍDA (Label 1)
        caminho_saidas = os.path.join(root_dir, 'database', 'SAIDAS_EQX', '*.JPG')
        for path in glob(caminho_saidas):
            self.image_paths.append(path)
            self.labels.append(1)

        print(f"✅ Concluído! Total de imagens encontradas: {len(self.image_paths)}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            # Albumentations requer arrays numpy
            image_np = np.array(image)
            augmented = self.transform(image=image_np)
            image = augmented['image']

        return image, label

In [ ]:
import random
import os
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader

# DEFINIÇÃO DO TAMANHO DA IMAGEM PARA KAN PURA (Evitar estouro de memória)
IMG_SIZE = 64

# Transformações com Albumentations (Prevenção de Overfitting + Variabilidade Forense)
transform_treino = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE), # Modificado aqui
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=30, p=0.5),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Transformações puras para o TESTE
transform_teste = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE), # Modificado aqui
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# 1. Configuração de Sementes
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# 2. Instanciação Dupla
dataset_base_treino = GunshotDataset(root_dir='./FDCPUnBGunshotDB', transform=transform_treino)
dataset_base_teste  = GunshotDataset(root_dir='./FDCPUnBGunshotDB', transform=transform_teste)

# 3. Separação Semântica (Por Caso)
case_to_indices = {}
for idx, path in enumerate(dataset_base_treino.image_paths):
    filename = os.path.basename(path)
    case_id = filename[:9]
    if case_id not in case_to_indices:
        case_to_indices[case_id] = []
    case_to_indices[case_id].append(idx)

unique_cases = list(case_to_indices.keys())
cases_treino, cases_teste = train_test_split(unique_cases, test_size=0.2, random_state=RANDOM_SEED)

# 4. Mapeamento de índices
indices_treino = []
for case in cases_treino:
    indices_treino.extend(case_to_indices[case])

indices_teste = []
for case in cases_teste:
    indices_teste.extend(case_to_indices[case])

# 5. Subsets e DataLoaders
dataset_treino = Subset(dataset_base_treino, indices_treino)
dataset_teste  = Subset(dataset_base_teste, indices_teste)

trainloader = DataLoader(dataset_treino, batch_size=32, shuffle=True)
testloader  = DataLoader(dataset_teste, batch_size=32, shuffle=False)

print("\n" + "="*50)
print("🚀 SANITY CHECK DE VAZAMENTO DE DADOS (CÓDIGO)")
print("="*50)

train_cases_check = set([os.path.basename(dataset_base_treino.image_paths[i])[:9] for i in indices_treino])
test_cases_check  = set([os.path.basename(dataset_base_teste.image_paths[i])[:9] for i in indices_teste])
intersecao = train_cases_check.intersection(test_cases_check)

print(f"📊 Total de Casos Únicos: {len(unique_cases)}")
print(f"   -> Casos no Treino: {len(train_cases_check)} casos ({len(dataset_treino)} imagens)")
print(f"   -> Casos no Teste:  {len(test_cases_check)} casos ({len(dataset_teste)} imagens)")

if len(intersecao) == 0:
    print("\n✅ SUCESSO! Zero interseção semântica identificada.")
else:
    print(f"\n❌ ALERTA DE VAZAMENTO! {len(intersecao)} casos vazados: {intersecao}")
print("="*50)

🔄 Mapeando imagens da base de dados...
✅ Concluído! Total de imagens encontradas: 2554
🔄 Mapeando imagens da base de dados...
✅ Concluído! Total de imagens encontradas: 2554

🚀 SANITY CHECK DE VAZAMENTO DE DADOS (CÓDIGO)
📊 Total de Casos Únicos: 486
   -> Casos no Treino: 388 casos (2099 imagens)
   -> Casos no Teste:  98 casos (455 imagens)

✅ SUCESSO! Zero interseção semântica identificada.


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_10821/1307369946.py:19: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
/tmp/ipykernel_10821/1307369946.py:20: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),


In [ ]:
class PureKAN_Gunshot(nn.Module):
    def __init__(self, img_size=64):
        super(PureKAN_Gunshot, self).__init__()

        # O tamanho da entrada será Canais (3) x Altura x Largura
        self.flat_size = 3 * img_size * img_size

        self.dropout = nn.Dropout(p=0.5)

        # Arquitetura KAN: [Entrada, Oculta 1, Oculta 2, Saída]
        # Adicionei uma camada intermediária a mais para lidar com a complexidade dos pixels
        self.kan = KAN([self.flat_size, 256, 64, 2], grid_size=5, spline_order=3)

    def forward(self, x):
        # Flatten: Transforma a imagem 2D em um vetor 1D
        # De (Batch, 3, 64, 64) para (Batch, 12288)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.kan(x)
        return x

model_gunshot = PureKAN_Gunshot(img_size=IMG_SIZE).to(device)
print(f"🚀 Modelo PureKAN_Gunshot inicializado e enviado para: {device.type.upper()}")

🚀 Modelo PureKAN_Gunshot inicializado e enviado para: CUDA


In [ ]:
# ====================================================
# 1. CÁLCULO DINÂMICO DE PESOS PARA O LOSS
# ====================================================
labels_treino = [dataset_base_treino.labels[i] for i in indices_treino]
num_entradas = labels_treino.count(0)
num_saidas = labels_treino.count(1)
total_amostras = len(labels_treino)

peso_entrada = total_amostras / (2.0 * num_entradas)
peso_saida = total_amostras / (2.0 * num_saidas)

pesos_classes = torch.tensor([peso_entrada, peso_saida], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=pesos_classes)

print(f"⚖️ Pesos de Classe Aplicados - Entrada (0): {peso_entrada:.2f} | Saída (1): {peso_saida:.2f}")

# ====================================================
# 2. CONFIGURAÇÃO DO TREINAMENTO
# ====================================================
total_epochs = 60 # Treinamento contínuo sem fases

best_acc = 0.0
best_model_wts = copy.deepcopy(model_gunshot.state_dict())

optimizer = torch.optim.AdamW(model_gunshot.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)

print(f"🔥 Iniciando Treinamento da KAN Pura ({total_epochs} épocas no total)...")

for epoch in range(total_epochs):
    # ================= FASE DE TREINO =================
    model_gunshot.train()
    running_loss, correct_train, total_train = 0.0, 0, 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_gunshot(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        running_loss += loss.item()

    acc_train = 100 * correct_train / total_train

    # ================= FASE DE TESTE =================
    model_gunshot.eval()
    correct_test, total_test = 0, 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_gunshot(images)

            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    acc_test = 100 * correct_test / total_test
    scheduler.step()

    # ================= CHECKPOINT =================
    if acc_test > best_acc:
        best_acc = acc_test
        best_model_wts = copy.deepcopy(model_gunshot.state_dict())
        torch.save(best_model_wts, 'melhor_modelo_tiros_purekan.pth')
        indicador_recorde = "🏆 NOVO RECORDE!"
    else:
        indicador_recorde = ""

    print(f"Época {epoch+1:03d}/{total_epochs} | Treino: {acc_train:.2f}% (Loss: {running_loss/len(trainloader):.4f}) | Teste: {acc_test:.2f}% {indicador_recorde}")

print(f"\n✅ Treinamento concluído. Melhor Acurácia de Validação: {best_acc:.2f}%")

⚖️ Pesos de Classe Aplicados - Entrada (0): 0.68 | Saída (1): 1.92
🔥 Iniciando Treinamento da KAN Pura (60 épocas no total)...
Época 001/60 | Treino: 53.60% (Loss: 0.7048) | Teste: 49.89% 🏆 NOVO RECORDE!
Época 002/60 | Treino: 60.51% (Loss: 0.6844) | Teste: 48.57% 
Época 003/60 | Treino: 59.55% (Loss: 0.6707) | Teste: 69.23% 🏆 NOVO RECORDE!
Época 004/60 | Treino: 60.12% (Loss: 0.6692) | Teste: 55.82% 
Época 005/60 | Treino: 61.03% (Loss: 0.6596) | Teste: 67.25% 
Época 006/60 | Treino: 61.36% (Loss: 0.6624) | Teste: 40.00% 
Época 007/60 | Treino: 61.70% (Loss: 0.6645) | Teste: 48.79% 
Época 008/60 | Treino: 62.17% (Loss: 0.6537) | Teste: 49.01% 
Época 009/60 | Treino: 60.36% (Loss: 0.6494) | Teste: 47.03% 
Época 010/60 | Treino: 59.98% (Loss: 0.6371) | Teste: 48.79% 
Época 011/60 | Treino: 59.41% (Loss: 0.6501) | Teste: 46.37% 
Época 012/60 | Treino: 60.74% (Loss: 0.6435) | Teste: 57.80% 
Época 013/60 | Treino: 63.03% (Loss: 0.6456) | Teste: 43.96% 
Época 014/60 | Treino: 63.17% (Loss: 

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

print("📥 Carregando o estado ótimo do modelo para geração de métricas...")
model_gunshot.load_state_dict(torch.load('melhor_modelo_tiros_purekan.pth'))
model_gunshot.eval()

y_true = []
y_pred = []
y_prob_both = []

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        outputs = model_gunshot(images)

        probabilities = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())
        y_prob_both.extend(probabilities.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob_both = np.array(y_prob_both)

y_true_onehot = np.eye(2)[y_true]

# Extração de variáveis da Matriz de Confusão
cm = confusion_matrix(y_true, y_pred)
FP = cm.sum(axis=0) - np.diag(cm)
FN = cm.sum(axis=1) - np.diag(cm)
TP = np.diag(cm)
TN = cm.sum() - (FP + FN + TP)
support_per_class = cm.sum(axis=1)

# Cálculo de Métricas
acc = accuracy_score(y_true, y_pred)
metrics = {}
averages = ['macro', 'micro', 'weighted']

for avg in averages:
    metrics[f'Prec_{avg}'] = precision_score(y_true, y_pred, average=avg, zero_division=0)
    metrics[f'Rec_{avg}'] = recall_score(y_true, y_pred, average=avg, zero_division=0)
    metrics[f'F1_{avg}'] = f1_score(y_true, y_pred, average=avg, zero_division=0)
    metrics[f'AUC_{avg}'] = roc_auc_score(y_true_onehot, y_prob_both, average=avg)

specificity_per_class = TN / (TN + FP)
metrics['Spec_macro'] = np.mean(specificity_per_class)
metrics['Spec_micro'] = TN.sum() / (TN.sum() + FP.sum())
metrics['Spec_weighted'] = np.average(specificity_per_class, weights=support_per_class)

# Geração da Tabela
print("\n" + "="*145)
print(" TABELA 5 - PERFORMANCE METRICS (FORMATO CIENTÍFICO)")
print("="*145)

header1 = f"{'Architecture':<14} {'Variant':<10} {'ACC':<7} "
header1 += f"{'Precision':<23} {'Recall':<23} {'F1-Score':<23} {'AUC':<23} {'Specificity':<23}"
print(header1)

header2 = f"{'':<33} "
for _ in range(5):
    header2 += f"{'M':<7} {'m':<7} {'W':<7}   "
print(header2)
print("-" * 145)

row = f"{'PureKAN':<14} {'Gunshot':<10} {acc:<7.3f} "
row += f"{metrics['Prec_macro']:<7.3f} {metrics['Prec_micro']:<7.3f} {metrics['Prec_weighted']:<7.3f}   "
row += f"{metrics['Rec_macro']:<7.3f} {metrics['Rec_micro']:<7.3f} {metrics['Rec_weighted']:<7.3f}   "
row += f"{metrics['F1_macro']:<7.3f} {metrics['F1_micro']:<7.3f} {metrics['F1_weighted']:<7.3f}   "
row += f"{metrics['AUC_macro']:<7.3f} {metrics['AUC_micro']:<7.3f} {metrics['AUC_weighted']:<7.3f}   "
row += f"{metrics['Spec_macro']:<7.3f} {metrics['Spec_micro']:<7.3f} {metrics['Spec_weighted']:<7.3f}"
print(row)
print("="*145 + "\n")

📥 Carregando o estado ótimo do modelo para geração de métricas...

 TABELA 5 - PERFORMANCE METRICS (FORMATO CIENTÍFICO)
Architecture   Variant    ACC     Precision               Recall                  F1-Score                AUC                     Specificity            
                                  M       m       W         M       m       W         M       m       W         M       m       W         M       m       W         
-------------------------------------------------------------------------------------------------------------------------------------------------
PureKAN        Gunshot    0.692   0.614   0.692   0.692     0.614   0.692   0.692     0.614   0.692   0.692     0.684   0.733   0.684     0.614   0.692   0.536  

